## Imports

**What I did:** Imported PyTorch (`torch`, `torch.nn`), PyTorch's pruning utilities (`torch.nn.utils.prune`), and the course-provided `helper_utils` module (which supplies `show_weights`, data loading, training, and reporting helpers used throughout this lab).

In [1]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

import helper_utils

**What I did:** Defined a small `SimpleModel` — one `Conv2d(3,16,kernel_size=3)` layer (`conv1`) and one `Linear(16*4*4, 10)` layer (`fc1`) — purely as a demonstration architecture for a 6×6 dummy input, so I can see pruning's mechanics clearly without the noise of a real training run.

In [2]:
# Create a simple model for demonstration
class SimpleModel(nn.Module):
    """
    A simple convolutional neural network defined for demonstration purposes.
    """
    def __init__(self):
        """
        Initializes the SimpleModel with a convolutional layer and a fully connected layer.
        """
        # Call the parent class initializer
        super(SimpleModel, self).__init__()
        # Define the first convolutional layer with 3 input channels and 16 output channels
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3)
        # Define the ReLU activation function
        self.relu1 = nn.ReLU()
        # Define the fully connected layer
        # The input size is calculated based on a 6x6 input image: (6-3+1) = 4x4 spatial output
        self.fc1 = nn.Linear(16 * 4 * 4, 10)  # Assuming input is 6x6

    def forward(self, x):
        """
        Defines the forward pass of the model.

        Args:
            x: The input tensor containing the image data.

        Returns:
            The output tensor after passing through the network.
        """
        # Apply the convolutional layer followed by the ReLU activation
        x = self.relu1(self.conv1(x))
        # Flatten the tensor dimensions for the fully connected layer
        x = x.view(x.size(0), -1)
        # Pass the flattened tensor through the fully connected layer
        x = self.fc1(x)
        return x

**What I did:** Instantiated a fresh `SimpleModel` to use as the target for **unstructured pruning**.

In [3]:
model = SimpleModel()

**What I did:** Printed the model's parameter names and buffer names *before* pruning — confirming `conv1`/`fc1` have their normal `weight`/`bias` parameters and that the buffers list is empty (no `weight_mask` yet).

In [4]:
# Check state before pruning
print("BEFORE UNSTRUCTURED PRUNING:\n")

# Model Parameters
print(f"Model parameters: {[name for name, _ in model.named_parameters()]}")
# Model Buffers
print(f"Model buffers: {[name for name, _ in model.named_buffers()]}")

BEFORE UNSTRUCTURED PRUNING:

Model parameters: ['conv1.weight', 'conv1.bias', 'fc1.weight', 'fc1.bias']
Model buffers: []


* Print out the raw numerical values of a kernel from the `conv1` layer.
    * This provides a direct, unambiguous baseline that you can compare against after the pruning is applied.

**What I did:** Called `helper_utils.show_weights(model, ['conv1'])` to print the raw numerical values of the `conv1` kernel before pruning, giving me a baseline to compare against afterward.

In [5]:
print("--- conv1 Weights Before Pruning ---\n")
helper_utils.show_weights(model, ['conv1'])

--- conv1 Weights Before Pruning ---

--- Weights for 'conv1' ---
[[ 0.14713551  0.1597351  -0.04508576 -0.1411868   0.16727689  0.03601883]
 [ 0.17678683 -0.0421665   0.03883464  0.14218384  0.02606363  0.09279715]
 [-0.0936953   0.1130226   0.16965301 -0.02717222  0.14835705  0.02844584]
 [-0.11571494  0.01816404 -0.19007894  0.02999349  0.15549262  0.02103858]
 [ 0.17380342 -0.16348024  0.14857674 -0.0606936   0.0517082  -0.05218851]
 [ 0.03202738 -0.06248942  0.11892453  0.08099408  0.17182371  0.11124761]]
--------------------------------------------------



**What I did:** Applied **unstructured pruning** with `prune.l1_unstructured(model.conv1, name="weight", amount=0.3)`, removing the 30% of individual weights in `conv1` with the smallest L1 (absolute) magnitude.

In [6]:
# Apply unstructured pruning to conv1
prune.l1_unstructured(model.conv1, name="weight", amount=0.3)

Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))

**What I did:** Checked the model's state after pruning — confirming `conv1.weight` was replaced by `conv1.weight_orig`, a new `conv1.weight_mask` buffer appeared, and printed the `conv1` weights again to see the scattered zeros left by pruning.

In [7]:
# Check state after pruning
print("AFTER UNSTRUCTURED PRUNING:\n")
print(f"Model parameters:", [name for name, _ in model.named_parameters()])
print(f"Model buffers:", [name for name, _ in model.named_buffers()])

# --- Show numerical weights after pruning ---
print("\n--- conv1 Weights After Pruning ---\n")
helper_utils.show_weights(model, ['conv1'])

AFTER UNSTRUCTURED PRUNING:

Model parameters: ['conv1.bias', 'conv1.weight_orig', 'fc1.weight', 'fc1.bias']
Model buffers: ['conv1.weight_mask']

--- conv1 Weights After Pruning ---

--- Weights for 'conv1' ---
[[ 0.14713551  0.1597351  -0.         -0.1411868   0.16727689  0.        ]
 [ 0.17678683 -0.          0.          0.14218384  0.          0.09279715]
 [-0.0936953   0.1130226   0.16965301 -0.          0.14835705  0.        ]
 [-0.11571494  0.         -0.19007894  0.          0.15549262  0.        ]
 [ 0.17380342 -0.16348024  0.14857674 -0.          0.         -0.        ]
 [ 0.         -0.06248942  0.11892453  0.08099408  0.17182371  0.11124761]]
--------------------------------------------------



**What I did:** Computed exact pruning statistics for `conv1` — counted zeroed entries in `weight_mask` against the total to confirm ~30% sparsity, and checked that `weight` is now a plain `Tensor` while `weight_orig` is the real, learnable `Parameter`.

In [8]:
# Check if the pruning mask exists.
if hasattr(model.conv1, 'weight_mask'):
    
    # Count weights where the mask is 0.
    pruned_count = torch.sum(model.conv1.weight_mask == 0).item()
    
    # Get the total number of weights.
    total_count = model.conv1.weight_mask.numel()
    
    # Print statistics.
    print(f"\nPruning statistics:")
    print(f"Total weights in conv1: {total_count}")
    print(f"Pruned weights (set to zero) in conv1: {pruned_count}")
    print(f"Sparsity: {pruned_count/total_count:.2%}")

    # Verify the change in weight attribute types.
    print(f"\nWeight type: {type(model.conv1.weight)}")
    print(f"Weight_orig type: {type(model.conv1.weight_orig)}")
else:
    # Handle pruning failure.
    print("Pruning did not work as expected! No weight_mask found.")


Pruning statistics:
Total weights in conv1: 432
Pruned weights (set to zero) in conv1: 130
Sparsity: 30.09%

Weight type: <class 'torch.Tensor'>
Weight_orig type: <class 'torch.nn.parameter.Parameter'>


**What I did:** Called `prune.remove(model.conv1, 'weight')` to make the unstructured pruning permanent — this bakes the sparse weights into a standard `.weight` tensor and discards the temporary `weight_mask`/`weight_orig` scaffolding.

In [9]:
# Make pruning permanent for conv1
prune.remove(model.conv1, 'weight')

Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))

**What I did:** Verified that `prune.remove()` worked as expected — `conv1.weight_orig` is gone, `conv1.weight` is back as a normal parameter, the buffers list is empty again, and `prune.is_pruned(model.conv1)` returns `False`.

In [10]:
# Check state to see if pruning is permanent
print("AFTER MAKING UNSTRUCTURED PRUNING PERMANENT:\n")

print(f"Model parameters: {[name for name, _ in model.named_parameters()]}")
print(f"Model buffers: {[name for name, _ in model.named_buffers()]}")
print(f"\nIs conv1 still considered pruned? {prune.is_pruned(model.conv1)}\n")

helper_utils.show_weights(model, ['conv1'])

AFTER MAKING UNSTRUCTURED PRUNING PERMANENT:

Model parameters: ['conv1.bias', 'conv1.weight', 'fc1.weight', 'fc1.bias']
Model buffers: []

Is conv1 still considered pruned? False

--- Weights for 'conv1' ---
[[ 0.14713551  0.1597351  -0.         -0.1411868   0.16727689  0.        ]
 [ 0.17678683 -0.          0.          0.14218384  0.          0.09279715]
 [-0.0936953   0.1130226   0.16965301 -0.          0.14835705  0.        ]
 [-0.11571494  0.         -0.19007894  0.          0.15549262  0.        ]
 [ 0.17380342 -0.16348024  0.14857674 -0.          0.         -0.        ]
 [ 0.         -0.06248942  0.11892453  0.08099408  0.17182371  0.11124761]]
--------------------------------------------------



**What I did:** Re-initialized a fresh `SimpleModel` to demonstrate **structured pruning** on a clean, unpruned instance.

In [11]:
model = SimpleModel()

* Next, you'll inspect the initial state of the `fc1` layer's weights before applying any changes.

**What I did:** Printed the model's parameters/buffers and the `fc1` layer's raw weights before applying structured pruning, as a baseline for comparison.

In [12]:
# Check state before pruning
print("BEFORE STRUCTURED PRUNING:\n")
# Model State
print(f"Model parameters: {[name for name, _ in model.named_parameters()]}")
print(f"Model buffers: {[name for name, _ in model.named_buffers()]}")

print("\n--- fc1 Weights Before Pruning ---\n")
helper_utils.show_weights(model, ['fc1'])

BEFORE STRUCTURED PRUNING:

Model parameters: ['conv1.weight', 'conv1.bias', 'fc1.weight', 'fc1.bias']
Model buffers: []

--- fc1 Weights Before Pruning ---

--- Weights for 'fc1' ---
[[-0.05283184 -0.00934087 -0.00476791  0.0344961   0.05883493  0.01604985]
 [ 0.05967887  0.03596281 -0.00977938  0.05441474  0.00336138  0.03949467]
 [ 0.03176349  0.03743436  0.05891279  0.02046806  0.02256147  0.01965842]
 [ 0.00203661  0.03068814 -0.01664933 -0.01753412  0.04982399 -0.04789251]
 [-0.00123788  0.01990238 -0.02599093  0.01133191  0.04145255 -0.01529047]
 [-0.05470549 -0.01276848 -0.02756809  0.04616866 -0.00030239  0.04859978]]
--------------------------------------------------



**What I did:** Applied **structured pruning** with `prune.ln_structured(model.fc1, name="weight", amount=0.5, n=2, dim=0)`, removing 50% of `fc1`'s output neurons (entire rows of the weight matrix) based on their L2 norm.

In [13]:
# Apply structured pruning to fc1
prune.ln_structured(model.fc1, name="weight", amount=0.5, n=2, dim=0)

Linear(in_features=256, out_features=10, bias=True)

**What I did:** Checked the model's state after structured pruning — `fc1.weight_orig` and a `fc1.weight_mask` buffer appeared, and printed `fc1`'s weights to confirm entire rows are now all zero (not scattered individual values like unstructured pruning).

In [14]:
# Check state after pruning
print("AFTER STRUCTURED PRUNING:\n")
print(f"Model parameters:", [name for name, _ in model.named_parameters()])
print(f"Model buffers:", [name for name, _ in model.named_buffers()])

# --- Show numerical weights after pruning ---
print("\n--- fc1 Weights After Pruning ---\n")
helper_utils.show_weights(model, ['fc1'])

AFTER STRUCTURED PRUNING:

Model parameters: ['conv1.weight', 'conv1.bias', 'fc1.bias', 'fc1.weight_orig']
Model buffers: ['fc1.weight_mask']

--- fc1 Weights After Pruning ---

--- Weights for 'fc1' ---
[[-0.05283184 -0.00934087 -0.00476791  0.0344961   0.05883493  0.01604985]
 [ 0.          0.         -0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.        ]
 [ 0.00203661  0.03068814 -0.01664933 -0.01753412  0.04982399 -0.04789251]
 [-0.          0.         -0.          0.          0.         -0.        ]
 [-0.05470549 -0.01276848 -0.02756809  0.04616866 -0.00030239  0.04859978]]
--------------------------------------------------



* This check calculates the pruning statistics.
    * Note that instead of counting individual zero-weights, this code now counts how many entire rows are zero to confirm that 50% of the output neurons were removed.

**What I did:** Calculated structured-pruning statistics for `fc1` by counting how many entire output-neuron rows are all-zero, confirming 50% of neurons (5 out of 10) were removed.

In [15]:
# Check if the pruning mask exists for fc1.
if hasattr(model.fc1, 'weight_mask'):
    # For structured pruning, you count entire rows of zeros.
    # Determine if each row in the weight tensor is all zeros.
    zero_rows = torch.sum(model.fc1.weight == 0, dim=1) == model.fc1.weight.shape[1]
    # Count the number of all-zero rows.
    pruned_count = torch.sum(zero_rows).item()
    # Get the total number of rows (output neurons).
    total_count = model.fc1.weight.shape[0]
    
    # Print the statistics.
    print(f"\nPruning statistics:\n")
    print(f"Total output neurons in fc1: {total_count}")
    print(f"Pruned output neurons (set to zero) in fc1: {pruned_count}")
    print(f"Sparsity: {pruned_count/total_count:.2%}")
    
    # Verify the change in attribute types.
    print(f"\nWeight type: {type(model.fc1.weight)}")
    print(f"Weight_orig type: {type(model.fc1.weight_orig)}")
else:
    # Handle pruning failure.
    print("\nStructured pruning did not work as expected!")


Pruning statistics:

Total output neurons in fc1: 10
Pruned output neurons (set to zero) in fc1: 5
Sparsity: 50.00%

Weight type: <class 'torch.Tensor'>
Weight_orig type: <class 'torch.nn.parameter.Parameter'>


**What I did:** Called `prune.remove(model.fc1, 'weight')` to make the structured pruning permanent, folding the sparse rows into a standard `.weight` tensor.

In [16]:
# Make structured pruning permanent for fc1
prune.remove(model.fc1, 'weight')

Linear(in_features=256, out_features=10, bias=True)

* This final check confirms the model's parameters and buffers have returned to their normal state and that the `fc1` layer is no longer considered "pruned" by PyTorch.

**What I did:** Confirmed the model returned to a normal state after `prune.remove()` — `fc1.weight_orig` is gone, `fc1.weight` is restored, buffers are empty, and `prune.is_pruned(model.fc1)` is `False`.

In [17]:
# Check if structured pruning is permanent
print("AFTER MAKING STRUCTURED PRUNING PERMANENT:\n")
print(f"Model parameters: {[name for name, _ in model.named_parameters()]}")
print(f"Model buffers: {[name for name, _ in model.named_buffers()]}")
print(f"\nIs fc1 still considered pruned? {prune.is_pruned(model.fc1)}\n")

helper_utils.show_weights(model, ['fc1'])

AFTER MAKING STRUCTURED PRUNING PERMANENT:

Model parameters: ['conv1.weight', 'conv1.bias', 'fc1.bias', 'fc1.weight']
Model buffers: []

Is fc1 still considered pruned? False

--- Weights for 'fc1' ---
[[-0.05283184 -0.00934087 -0.00476791  0.0344961   0.05883493  0.01604985]
 [ 0.          0.         -0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.        ]
 [ 0.00203661  0.03068814 -0.01664933 -0.01753412  0.04982399 -0.04789251]
 [-0.          0.         -0.          0.          0.         -0.        ]
 [-0.05470549 -0.01276848 -0.02756809  0.04616866 -0.00030239  0.04859978]]
--------------------------------------------------



**What I did:** Re-initialized a fresh `SimpleModel` again, this time to demonstrate **global pruning** across multiple layers at once.

In [18]:
model = SimpleModel()

* Next, you'll inspect the initial state of both the `conv1` and `fc1` layers before applying any changes.

**What I did:** Printed the model's parameters/buffers and the initial weights of both `conv1` and `fc1` before applying global pruning.

In [19]:
# Check the model's initial state before pruning.
print("BEFORE GLOBAL PRUNING:\n")

print(f"Model parameters: {[name for name, _ in model.named_parameters()]}")
print(f"Model buffers: {[name for name, _ in model.named_buffers()]}")

# Show the initial weights of both layers.
print("\n--- Initial Weights of conv1 and fc1 ---\n")
helper_utils.show_weights(model, ['conv1', 'fc1'])

BEFORE GLOBAL PRUNING:

Model parameters: ['conv1.weight', 'conv1.bias', 'fc1.weight', 'fc1.bias']
Model buffers: []

--- Initial Weights of conv1 and fc1 ---

--- Weights for 'conv1' ---
[[-0.00060812 -0.09019063  0.05485618 -0.01905923 -0.15058206 -0.01522618]
 [-0.15441509 -0.1331244  -0.00638882 -0.00541645  0.03185495  0.0861361 ]
 [ 0.11097497 -0.05141376 -0.01014832  0.05731362  0.10732383 -0.03668859]
 [-0.18135874 -0.09984754  0.1471441   0.03727537 -0.12656902  0.1825963 ]
 [ 0.17376752  0.01255421 -0.14229308  0.1689901  -0.08442686 -0.10398789]
 [-0.09045125  0.0914873   0.17926638  0.11064124 -0.13907064  0.16064169]]
--------------------------------------------------

--- Weights for 'fc1' ---
[[-0.0035399   0.03981163 -0.02674621 -0.00782842  0.031175    0.05252821]
 [ 0.06139955  0.01319046 -0.01279566  0.04805665  0.030821   -0.04201072]
 [-0.01958526 -0.04838447  0.0351347   0.03984992 -0.01887447 -0.00583459]
 [-0.04159155  0.01379046  0.04890917  0.02150661  0.05053

**What I did:** Built a `parameters_to_prune` tuple listing `(model.conv1, 'weight')` and `(model.fc1, 'weight')`, then called `prune.global_unstructured(...)` with `pruning_method=prune.L1Unstructured, amount=0.2` to remove the weakest 20% of weights across *both* layers combined (not 20% from each layer individually).

In [20]:
# Define the collection of parameters to be pruned globally.
parameters_to_prune = (
    (model.conv1, 'weight'),
    (model.fc1, 'weight'),
)

# Apply global unstructured pruning across the specified parameters.
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.2,
)

**What I did:** Checked the model's state after global pruning — `_orig`/`_mask` attributes now exist for both `conv1` and `fc1`, and printed both layers' weights to see the scattered zeros.

In [21]:
# Verify the model's state after pruning.
print("AFTER GLOBAL PRUNING:\n")

print(f"Model parameters:", [name for name, _ in model.named_parameters()])
print(f"Model buffers:", [name for name, _ in model.named_buffers()])

print("\n--- Weights After Global Pruning ---\n")
helper_utils.show_weights(model, ['conv1', 'fc1'])

AFTER GLOBAL PRUNING:

Model parameters: ['conv1.bias', 'conv1.weight_orig', 'fc1.bias', 'fc1.weight_orig']
Model buffers: ['conv1.weight_mask', 'fc1.weight_mask']

--- Weights After Global Pruning ---

--- Weights for 'conv1' ---
[[-0.         -0.09019063  0.05485618 -0.01905923 -0.15058206 -0.01522618]
 [-0.15441509 -0.1331244  -0.         -0.          0.03185495  0.0861361 ]
 [ 0.11097497 -0.05141376 -0.          0.05731362  0.10732383 -0.03668859]
 [-0.18135874 -0.09984754  0.1471441   0.03727537 -0.12656902  0.1825963 ]
 [ 0.17376752  0.         -0.14229308  0.1689901  -0.08442686 -0.10398789]
 [-0.09045125  0.0914873   0.17926638  0.11064124 -0.13907064  0.16064169]]
--------------------------------------------------

--- Weights for 'fc1' ---
[[-0.          0.03981163 -0.02674621 -0.          0.031175    0.05252821]
 [ 0.06139955  0.         -0.          0.04805665  0.030821   -0.04201072]
 [-0.01958526 -0.04838447  0.0351347   0.03984992 -0.01887447 -0.        ]
 [-0.04159155  

**What I did:** Computed sparsity separately for `conv1` and `fc1` to show that global pruning doesn't split the 20% evenly — each layer ends up with a different sparsity level depending on which weights were globally weakest.

In [22]:
# Check global pruning results.
print("--- Sparsity per Layer After Global Pruning ---\n")

# Verify sparsity in each layer individually.
conv1_sparsity = 100. * float(torch.sum(model.conv1.weight == 0)) / float(model.conv1.weight.nelement())
fc1_sparsity = 100. * float(torch.sum(model.fc1.weight == 0)) / float(model.fc1.weight.nelement())

print(f"Sparsity in conv1: {conv1_sparsity:.2f}%")
print(f"Sparsity in fc1: {fc1_sparsity:.2f}%")

--- Sparsity per Layer After Global Pruning ---

Sparsity in conv1: 8.33%
Sparsity in fc1: 21.95%


**What I did:** Looped over `parameters_to_prune` and called `prune.remove()` on each layer, making the global pruning permanent for both `conv1` and `fc1`.

In [23]:
# Make the pruning permanent for all pruned layers by iterating through the list.
for module, param_name in parameters_to_prune:
    prune.remove(module, param_name)

* This final check confirms that all pruned layers have been restored to their standard structure.

**What I did:** Did a final check confirming both `conv1` and `fc1` returned to their standard (non-pruned) structure, with `prune.is_pruned()` returning `False` for both and the zeroed weights preserved.

In [24]:
print("\nAFTER MAKING GLOBAL PRUNING PERMANENT:\n")

print(f"Model parameters: {[name for name, _ in model.named_parameters()]}")
print(f"Model buffers: {[name for name, _ in model.named_buffers()]}")

print(f"\nIs conv1 still considered pruned? {prune.is_pruned(model.conv1)}")
print(f"Is fc1 still considered pruned? {prune.is_pruned(model.fc1)}\n")

helper_utils.show_weights(model, ['conv1', 'fc1'])


AFTER MAKING GLOBAL PRUNING PERMANENT:

Model parameters: ['conv1.bias', 'conv1.weight', 'fc1.bias', 'fc1.weight']
Model buffers: []

Is conv1 still considered pruned? False
Is fc1 still considered pruned? False

--- Weights for 'conv1' ---
[[-0.         -0.09019063  0.05485618 -0.01905923 -0.15058206 -0.01522618]
 [-0.15441509 -0.1331244  -0.         -0.          0.03185495  0.0861361 ]
 [ 0.11097497 -0.05141376 -0.          0.05731362  0.10732383 -0.03668859]
 [-0.18135874 -0.09984754  0.1471441   0.03727537 -0.12656902  0.1825963 ]
 [ 0.17376752  0.         -0.14229308  0.1689901  -0.08442686 -0.10398789]
 [-0.09045125  0.0914873   0.17926638  0.11064124 -0.13907064  0.16064169]]
--------------------------------------------------

--- Weights for 'fc1' ---
[[-0.          0.03981163 -0.02674621 -0.          0.031175    0.05252821]
 [ 0.06139955  0.         -0.          0.04805665  0.030821   -0.04201072]
 [-0.01958526 -0.04838447  0.0351347   0.03984992 -0.01887447 -0.        ]
 [-0

## Dataset and Training Configuration

* Configure `DEVICE` to use a CUDA-enabled GPU if available, otherwise it defaults to the CPU.

**What I did:** Set `DEVICE` to `cuda` if a GPU is available, otherwise `cpu`, for the upcoming ResNet18 training section.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {DEVICE}")

**What I did:** Set `dataset_path` to the local `./fruit_and_vegetable_subset` folder — the Fruit and Vegetable Disease (Healthy vs Rotten) subset (28 classes, 200 images each) used for the practical ResNet18 pruning comparison.

In [ ]:
dataset_path = "./fruit_and_vegetable_subset"

* Create data loaders for model training.

**What I did:** Called `helper_utils.get_dataloaders(dataset_path)` to build an 80/20 train/validation split with the appropriate train/val transforms, returning `train_loader` and `val_loader`.

In [ ]:
train_loader, val_loader = helper_utils.get_dataloaders(dataset_path)

* Set the number of classes.

**What I did:** Read `num_classes` dynamically off the train loader's underlying dataset so the model's final layer can be sized correctly.

In [ ]:
# Get the number of classes dynamically from the train_loader's dataset
num_classes = len(train_loader.dataset.subset.dataset.classes)
print("Number of classes:", num_classes)

**What I did:** Set `num_epochs = 2` for both the unpruned-baseline and pruned-model training runs (kept identical so the comparison is fair).

In [ ]:
num_epochs = 2

**What I did:** Loaded a ResNet18 pretrained on ImageNet via `helper_utils.load_resnet18()`, then called `helper_utils.replace_final_layer()` to swap its final FC layer for one matching my `num_classes`, with no layers frozen — this is the unpruned baseline model.

In [ ]:
# Load the Resnet18 model with ImageNet weights
resnet18_model_unpruned = helper_utils.load_resnet18()

# Adapt the model for transfer learning with the new number of classes
model_unpruned = helper_utils.replace_final_layer(resnet18_model_unpruned, num_classes)

**What I did:** Ran `helper_utils.training_loop()` to fine-tune the unpruned ResNet18 for `num_epochs` epochs, getting back the trained model and its final accuracy/precision/recall/F1 metrics.

In [ ]:
# Train the unpruned model
trained_unpruned_model, unpruned_metrics = helper_utils.training_loop(model_unpruned,
                                                                      train_loader,
                                                                      val_loader,
                                                                      num_epochs,
                                                                      DEVICE,
                                                                      num_classes
                                                                     )

* Save the state dictionary and metrics of the trained unpruned model to file.

**What I did:** Called `helper_utils.save_unpruned_model_and_metrics()` to save the trained unpruned model's state dict and metrics to disk (`unpruned_model_state_dict.pth`, `unpruned_metrics.pkl`) for later comparison.

In [ ]:
helper_utils.save_unpruned_model_and_metrics(trained_unpruned_model, unpruned_metrics)

* Print the key performance metrics of the unpruned model after training is complete.

**What I did:** Printed the unpruned model's final accuracy, precision, recall, and F1-score after training.

In [ ]:
print(f"""
-- After Training for {num_epochs} Epoch(s) --
Final Accuracy:           {unpruned_metrics['accuracy']:.2f}%
Final Precision (Macro):  {unpruned_metrics['precision']:.4f}
Final Recall (Macro):     {unpruned_metrics['recall']:.4f}
Final F1-Score (Macro):   {unpruned_metrics['f1_score']:.4f}""")

**What I did:** Loaded a second, fresh ResNet18 with ImageNet weights and replaced its final layer the same way as the baseline — this is the starting point for the pruned model, so the comparison is apples-to-apples.

In [ ]:
# Load the Resnet18 model with ImageNet weights
resnet18_model_pruned = helper_utils.load_resnet18()

# Adapt the model for transfer learning with the new number of classes
model_pruned = helper_utils.replace_final_layer(resnet18_model_pruned, num_classes)

**What I did:** Defined `analyze_model_sparsity(model)`, a helper that loops over every module with a `weight` attribute, sums total vs. zero elements across the whole model, and prints overall sparsity — much faster than checking each ResNet18 layer individually the way I did for `SimpleModel`.

In [ ]:
def analyze_model_sparsity(model):
    """
    Calculates and prints the sparsity of a PyTorch model
    by inspecting the computed 'weight' attribute of each module.
    """
    # Initialize total parameter count for weighted layers.
    total_params = 0
    # Initialize total count of zero parameters.
    total_zero_params = 0

    # Loop through all named modules (layers) in the model.
    for name, module in model.named_modules():
        # Check if the module has a 'weight' attribute and if it's a PyTorch tensor.
        if hasattr(module, 'weight') and isinstance(module.weight, torch.Tensor):
            # Add the number of elements in the current module's weight tensor to total_params.
            total_params += module.weight.nelement()
            # Count and add the number of zero elements in the current weight tensor to total_zero_params.
            # Note: module.weight reflects the actual (masked) weights after pruning.
            total_zero_params += torch.sum(module.weight == 0).item()

    # Check if any weighted layers were found to avoid division by zero.
    if total_params > 0:
        # Calculate sparsity as a percentage.
        sparsity = 100. * float(total_zero_params) / float(total_params)
        # Print the calculated model sparsity.
        print(f"Model Sparsity: {sparsity:.2f}%")
        # Print the total number of parameters in weighted layers.
        print(f"Total parameters (in weighted layers): {total_params}")
        # Print the total count of zero parameters.
        print(f"Total zero parameters (in weighted layers): {total_zero_params}")
    else:
        # Inform if no weighted layers were found.
        print("No weighted layers found to analyze.")

**What I did:** Ran `analyze_model_sparsity(model_pruned)` before any pruning, confirming 0.00% sparsity (all weights non-zero) as a baseline for the full ResNet18.

In [ ]:
# --- Analysis Before Pruning ---
print("--- Analysis Before Pruning ---")
analyze_model_sparsity(model_pruned)

# # --- Show initial weights of selected layers ---
# print("\n--- Selected conv1 Layer Weights Before Pruning ---\n")
# helper_utils.show_weights(model_pruned, ['conv1'])

**What I did:** Collected every `Conv2d`/`Linear` layer in `model_pruned` *except* the final classifier (`fc`) into `parameters_to_prune`, then applied `prune.global_unstructured(..., amount=0.5)` — 50% global unstructured pruning across the whole feature-extraction backbone, deliberately leaving the newly-replaced classifier head untouched.

In [ ]:
# Identify all convolutional and linear layers to be pruned, excluding the final classifier
parameters_to_prune = []
for module_name, module in model_pruned.named_modules():
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)) and 'fc' not in module_name:
        parameters_to_prune.append((module, 'weight'))

# Apply global unstructured pruning
print("\n--- Applying 50% Global Unstructured Pruning ---\n")
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.5,
)
print("Global pruning applied.")

**What I did:** Re-ran `analyze_model_sparsity(model_pruned)` right after pruning (before making it permanent) and confirmed overall sparsity is now close to 50%, with the total parameter count unchanged — the weights are zeroed but not yet physically removed.

In [ ]:
# --- Analysis After Pruning (Before Making it Permanent) ---
print("--- Analysis After Pruning ---")
analyze_model_sparsity(model_pruned)

# # --- Show weights of selected layers after temporary pruning ---
# print("\n--- Selected Layer Weights After Temporary Pruning ---\n")
# helper_utils.show_weights(model_pruned, ['conv1'])

**What I did:** Ran `helper_utils.training_loop()` to fine-tune the pruned ResNet18 — done *before* `prune.remove()` so the optimizer updates `weight_orig` while `weight_mask` keeps the pruned connections locked at zero, letting the rest of the network recover accuracy around the pruned weights.

In [ ]:
# Train the pruned model
trained_pruned_model, pruned_metrics = helper_utils.training_loop(model_pruned,
                                                                  train_loader,
                                                                  val_loader,
                                                                  num_epochs,
                                                                  DEVICE,
                                                                  num_classes
                                                                 )

**What I did:** Called `prune.remove()` on every layer in `parameters_to_prune` to make the pruning permanent, then ran `analyze_model_sparsity()` one more time to confirm the zeroed weights persisted after finalizing.

In [ ]:
# --- Making Pruning Permanent ---
print("--- Making Pruning Permanent ---")
for module, param_name in parameters_to_prune:
    prune.remove(module, param_name)
print("Pruning has been made permanent.")

# --- Final Analysis ---
print("\n--- Final Analysis of Trained Model (Post-Permanent Pruning) ---")
analyze_model_sparsity(trained_pruned_model)

# # --- Show weights of selected layers after permanent pruning ---
# print("\n--- Selected Layer Weights After Permanent Pruning ---\n")
# helper_utils.show_weights(trained_pruned_model, ['conv1']) # Confirming persistence of zeros

* Save the state dictionary and metrics of the trained pruned model to file.

**What I did:** Called `helper_utils.save_pruned_model_and_metrics()` to save the trained, permanently-pruned model's state dict and metrics to disk for the final comparison.

In [ ]:
helper_utils.save_pruned_model_and_metrics(trained_pruned_model, pruned_metrics)

**What I did:** Called `helper_utils.comparison_report()`, loading both saved state dicts and metrics files to print a side-by-side report of total vs. non-zero parameters, saved file size, and accuracy/precision/recall/F1 for the unpruned vs. pruned ResNet18.

In [ ]:
helper_utils.comparison_report(
    unpruned_state_dict_path="unpruned_model_state_dict.pth",
    unpruned_metrics_path="unpruned_metrics.pkl",
    pruned_state_dict_path="pruned_model_permanent_state_dict.pth",
    pruned_metrics_path="pruned_metrics.pkl",
    num_epochs=num_epochs,
    device=DEVICE
)